# Heuristics comparison

This notebook compares different eviction policies and parameter settings for the single-node blobpool simulator.

## 1. Setup: generate test events

We create a reproducible set of test events to compare different configurations.

In [ ]:
import random
from dataclasses import dataclass

from single_node_sim import run
from single_node_sim.events import TxAnnouncement, BlockIncluded
from single_node_sim.params import HeuristicParams, EvictionPolicy, PRESETS
from single_node_sim.metrics import MetricsSummary, Role
from sparse_blobpool.protocol.constants import ALL_ONES

import matplotlib.pyplot as plt
import pandas as pd


def generate_test_events(
    num_transactions: int = 200,
    num_senders: int = 50,
    seed: int = 42,
    include_blocks: bool = True,
) -> list:
    """Generate a reproducible set of test events."""
    rng = random.Random(seed)
    events = []
    
    sender_nonces = {f"0xsender{i:04d}": 0 for i in range(num_senders)}
    all_hashes = []
    
    for i in range(num_transactions):
        timestamp = i * 0.1 + rng.uniform(0, 0.05)
        sender = rng.choice(list(sender_nonces.keys()))
        nonce = sender_nonces[sender]
        sender_nonces[sender] += 1
        
        tx_hash = f"0x{i:016x}"
        all_hashes.append(tx_hash)
        
        gas_fee_cap = rng.randint(500_000_000, 5_000_000_000)
        gas_tip_cap = rng.randint(50_000_000, min(gas_fee_cap, 500_000_000))
        blob_count = rng.choices([1, 2, 3, 4, 5, 6], weights=[50, 25, 12, 7, 4, 2])[0]
        tx_size = 131_072 * blob_count
        
        events.append(TxAnnouncement(
            timestamp=timestamp,
            tx_hash=tx_hash,
            sender=sender,
            nonce=nonce,
            gas_fee_cap=gas_fee_cap,
            gas_tip_cap=gas_tip_cap,
            tx_size=tx_size,
            blob_count=blob_count,
            cell_mask=ALL_ONES,
        ))
    
    if include_blocks:
        block_interval = 12.0
        current_time = block_interval
        max_time = num_transactions * 0.1 + 60.0
        
        while current_time < max_time:
            available_hashes = [
                h for h, e in zip(all_hashes, events)
                if isinstance(e, TxAnnouncement) and e.timestamp < current_time
            ]
            if available_hashes:
                included = rng.sample(
                    available_hashes,
                    min(len(available_hashes), rng.randint(1, 6))
                )
                events.append(BlockIncluded(
                    timestamp=current_time,
                    tx_hashes=included,
                ))
                for h in included:
                    all_hashes.remove(h)
            current_time += block_interval
    
    return sorted(events, key=lambda e: e.timestamp)


test_events = generate_test_events(num_transactions=200, seed=42)
print(f"Generated {len(test_events)} events")
print(f"  - TxAnnouncements: {sum(1 for e in test_events if isinstance(e, TxAnnouncement))}")
print(f"  - BlockIncluded: {sum(1 for e in test_events if isinstance(e, BlockIncluded))}")

## 2. Compare eviction policies (FEE_BASED vs AGE_BASED vs HYBRID)

We run the same events through different eviction policies and compare results.

In [ ]:
policies = {
    "FEE_BASED": EvictionPolicy.FEE_BASED,
    "AGE_BASED": EvictionPolicy.AGE_BASED,
    "HYBRID (0.3)": EvictionPolicy.HYBRID,
}

results = {}
for name, policy in policies.items():
    age_weight = 0.3 if "HYBRID" in name else 0.0
    params = HeuristicParams(
        max_pool_bytes=10 * 1024 * 1024,
        eviction_policy=policy,
        age_weight=age_weight,
        seed=42,
    )
    result = run(events=test_events, params=params)
    summary = result.metrics.summary()
    results[name] = (result, summary)
    print(f"\n{name}:")
    print(f"  Completions: {summary.total_completions}")
    print(f"  Evictions: {summary.total_evictions}")
    print(f"  Avg completion time: {summary.avg_completion_time:.2f}s")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics_to_plot = [
    ("total_completions", "Completions"),
    ("total_evictions", "Evictions"),
    ("avg_completion_time", "Avg completion time (s)"),
]

for ax, (metric, label) in zip(axes, metrics_to_plot):
    values = [getattr(results[name][1], metric) for name in policies]
    bars = ax.bar(list(policies.keys()), values, color=['steelblue', 'coral', 'mediumseagreen'])
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.annotate(f'{val:.1f}' if isinstance(val, float) else str(val),
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 3. Parameter sensitivity: vary provider_probability

Provider probability affects how many nodes store full data vs just sample.

In [ ]:
provider_probs = [0.05, 0.10, 0.15, 0.25, 0.40, 0.60]
provider_results = []

for p in provider_probs:
    params = HeuristicParams(
        provider_probability=p,
        seed=42,
    )
    result = run(events=test_events, params=params)
    summary = result.metrics.summary()
    
    records = result.metrics.get_tx_records()
    provider_count = sum(1 for r in records.values() if r.role == Role.PROVIDER)
    sampler_count = sum(1 for r in records.values() if r.role == Role.SAMPLER)
    
    provider_results.append({
        "probability": p,
        "providers": provider_count,
        "samplers": sampler_count,
        "observed_ratio": provider_count / len(records) if records else 0,
        "completions": summary.total_completions,
        "avg_time": summary.avg_completion_time,
    })

df_provider = pd.DataFrame(provider_results)
print(df_provider.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(df_provider['probability'], df_provider['observed_ratio'], 'o-', label='Observed', markersize=8)
ax1.plot([0, 0.6], [0, 0.6], 'k--', alpha=0.3, label='Expected')
ax1.set_xlabel('Provider probability (param)')
ax1.set_ylabel('Observed provider ratio')
ax1.set_title('Provider ratio vs parameter')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(range(len(provider_probs)), df_provider['completions'], color='steelblue')
ax2.set_xticks(range(len(provider_probs)))
ax2.set_xticklabels([f'{p:.0%}' for p in provider_probs])
ax2.set_xlabel('Provider probability')
ax2.set_ylabel('Total completions')
ax2.set_title('Completions by provider probability')

plt.tight_layout()
plt.show()

## 4. Parameter sensitivity: vary tx_ttl

Transaction TTL determines how long incomplete transactions stay in the pool.

In [ ]:
ttl_values = [15.0, 30.0, 60.0, 120.0, 300.0, 600.0]
ttl_results = []

for ttl in ttl_values:
    params = HeuristicParams(
        tx_ttl=ttl,
        seed=42,
    )
    result = run(events=test_events, params=params)
    summary = result.metrics.summary()
    
    records = result.metrics.get_tx_records()
    expired_count = sum(
        1 for r in records.values()
        if r.eviction_reason == "ttl_expired"
    )
    
    ttl_results.append({
        "ttl": ttl,
        "completions": summary.total_completions,
        "evictions": summary.total_evictions,
        "expired": expired_count,
        "avg_completion_time": summary.avg_completion_time,
        "peak_tx_count": summary.peak_tx_count,
    })

df_ttl = pd.DataFrame(ttl_results)
print(df_ttl.to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(df_ttl['ttl'], df_ttl['completions'], 'o-', color='green', markersize=8)
axes[0, 0].set_xlabel('TTL (seconds)')
axes[0, 0].set_ylabel('Completions')
axes[0, 0].set_title('Completions vs TTL')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(df_ttl['ttl'], df_ttl['expired'], 'o-', color='red', markersize=8)
axes[0, 1].set_xlabel('TTL (seconds)')
axes[0, 1].set_ylabel('Expired transactions')
axes[0, 1].set_title('TTL expirations vs TTL')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(df_ttl['ttl'], df_ttl['avg_completion_time'], 'o-', color='blue', markersize=8)
axes[1, 0].set_xlabel('TTL (seconds)')
axes[1, 0].set_ylabel('Avg completion time (s)')
axes[1, 0].set_title('Completion time vs TTL')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(df_ttl['ttl'], df_ttl['peak_tx_count'], 'o-', color='purple', markersize=8)
axes[1, 1].set_xlabel('TTL (seconds)')
axes[1, 1].set_ylabel('Peak tx count')
axes[1, 1].set_title('Peak pool size vs TTL')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Summary table of results

Compare all configurations in a summary table.

In [ ]:
all_results = []

for name, preset in PRESETS.items():
    result = run(events=test_events, params=preset)
    s = result.metrics.summary()
    all_results.append({
        "config": name,
        "announcements": s.total_announcements,
        "completions": s.total_completions,
        "evictions": s.total_evictions,
        "completion_rate": f"{100 * s.total_completions / s.total_announcements:.1f}%" if s.total_announcements > 0 else "N/A",
        "avg_time": f"{s.avg_completion_time:.2f}s",
        "peak_size_mb": f"{s.peak_pool_size / (1024*1024):.1f}",
        "peak_txs": s.peak_tx_count,
    })

df_summary = pd.DataFrame(all_results)
print("\nSummary of all preset configurations:")
print("=" * 100)
print(df_summary.to_string(index=False))
print("=" * 100)

print("\n\nKey observations:")
best_completion = df_summary.loc[df_summary['completions'].idxmax()]
print(f"- Best completion rate: {best_completion['config']} with {best_completion['completions']} completions")

lowest_eviction = df_summary.loc[df_summary['evictions'].idxmin()]
print(f"- Lowest evictions: {lowest_eviction['config']} with {lowest_eviction['evictions']} evictions")